**System initialized**

In [1]:
# ============================================================
# TBM-X — 01 BENCHMARK DATABASE
# ============================================================
# Amaç:
# Gerçek uçak benchmark verisini TBM-X tasarım sistemine
# hazırlamak.
#
# Raymer tabanlı fizik modeli + Data Science yaklaşımı
# burada veri tarafında birleşmeye başlıyor..
# ============================================================

print("TBM-X Benchmark Database")
print("=" * 50)
print("System initialized.")

TBM-X Benchmark Database
System initialized.


**Libraries**

In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

**PROJECT_ROOT**

In [3]:
# Notebook'un bulunduğu ana TBM-X klasörü
PROJECT_ROOT = Path.cwd()

print("Project root:")
print(PROJECT_ROOT)

Project root:
C:\Users\musta\Documents\RAYMER\TBM-X_v0.2_data_engineering


**DATASET**

In [4]:
# ============================================================
# RAW BENCHMARK DATASET
# ============================================================

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "TBM_X_Benchmark_Dataset_Seed_v0_4.csv"
)

df_raw = pd.read_csv(DATA_PATH)

print(f"Dataset loaded successfully.")
print(f"Rows    : {df_raw.shape[0]}")
print(f"Columns : {df_raw.shape[1]}")

Dataset loaded successfully.
Rows    : 47
Columns : 17


**First View**

In [5]:
df_raw.head(10)

,aircraft_id,manufacturer,variant,family,category,parameter,value,unit,status,confidence,source_id,source_type,condition,wind,payload,passenger_count,requirement_flag
0,TBM960,Daher,TBM 960,TBM,weights,MTOW,3454.00,kg,reported,A,S02,certification,CG-dependent,NaN,NaN,NaN,validation_reference
1,TBM960,Daher,TBM 960,TBM,weights,MRW,3470.00,kg,reported,A,S02,certification,CG-dependent,NaN,NaN,NaN,validation_reference
2,TBM960,Daher,TBM 960,TBM,weights,MZFW,2836.00,kg,reported,A,S05,manufacturer,6252 lb,NaN,NaN,NaN,validation_reference
3,TBM960,Daher,TBM 960,TBM,weights,empty_weight,2180.00,kg,reported,A,S05,manufacturer,Prestige cabin; 4806 lb,NaN,NaN,NaN,validation_reference
4,TBM960,Daher,TBM 960,TBM,fuel,usable_fuel,1106.00,L,reported,A,S05,manufacturer,292 US gal,NaN,NaN,NaN,validation_reference
5,TBM960,Daher,TBM 960,TBM,geometry,length,10.74,m,reported,A,S05,manufacturer,NaN,NaN,NaN,NaN,validation_reference
6,TBM960,Daher,TBM 960,TBM,geometry,span,12.83,m,reported,A,S05,manufacturer,NaN,NaN,NaN,NaN,validation_reference
7,TBM960,Daher,TBM 960,TBM,geometry,height,4.35,m,reported,A,S05,manufacturer,NaN,NaN,NaN,NaN,validation_reference
8,TBM960,Daher,TBM 960,TBM,geometry,wing_area,18.00,m2,reported,A,S02,certification,NaN,NaN,NaN,NaN,validation_reference
9,TBM960,Daher,TBM 960,TBM,performance,max_cruise_speed,330.00,KTAS,reported,A,S05,manufacturer,"28,000 ft",NaN,NaN,NaN,validation_reference


**Dataset columns**

In [6]:
print("Dataset columns:")
print("-" * 50)

for i, column in enumerate(df_raw.columns, start=1):
    print(f"{i:02d}. {column}")

Dataset columns:
--------------------------------------------------
01. aircraft_id
02. manufacturer
03. variant
04. family
05. category
06. parameter
07. value
08. unit
09. status
10. confidence
11. source_id
12. source_type
13. condition
14. wind
15. payload
16. passenger_count
17. requirement_flag


**Aircraft list**

In [7]:
print("Aircraft in benchmark database:")
print("-" * 50)

aircraft_list = df_raw["aircraft_id"].unique()

for aircraft in aircraft_list:
    print(aircraft)

Aircraft in benchmark database:
--------------------------------------------------
TBM960
TBM910
TBM940
PC12NGX
M600SLS


**Benchmark parameters**

In [8]:
print("Benchmark parameters:")
print("-" * 50)

parameters = (
    df_raw["parameter"]
    .dropna()
    .unique()
)

for parameter in parameters:
    print(parameter)
    

Benchmark parameters:
--------------------------------------------------
MTOW
MRW
MZFW
empty_weight
usable_fuel
length
span
height
wing_area
max_cruise_speed
ceiling
takeoff_distance_50ft
range
takeoff_power
nominal_power
diameter
rpm
basic_operating_weight
landing_distance_50ft
rate_of_climb
stall_speed_landing


**Missing values detection**

In [9]:
missing = pd.DataFrame({
    "column": df_raw.columns,
    "missing_count": df_raw.isna().sum().values,
    "missing_percent": (
        df_raw.isna().mean().values * 100
    )
})

missing.sort_values(
    "missing_percent",
    ascending=False
)

,column,missing_count,missing_percent
15,passenger_count,47,100.000000
14,payload,47,100.000000
13,wind,47,100.000000
12,condition,20,42.553191
0,aircraft_id,0,0.000000
9,confidence,0,0.000000
11,source_type,0,0.000000
10,source_id,0,0.000000
8,status,0,0.000000
1,manufacturer,0,0.000000


**Dublicate detection**

In [10]:
duplicate_keys = [
    "aircraft_id",
    "parameter",
    "unit",
    "source_id"
]

duplicates = df_raw[
    df_raw.duplicated(
        subset=duplicate_keys,
        keep=False
    )
].sort_values(duplicate_keys)

print(f"Duplicate records found: {len(duplicates)}")

duplicates.head(20)

Duplicate records found: 3


,aircraft_id,manufacturer,variant,family,category,parameter,value,unit,status,confidence,source_id,source_type,condition,wind,payload,passenger_count,requirement_flag
12,TBM960,Daher,TBM 960,TBM,performance,range,1730.0,NM,reported,A,S05,manufacturer,"252 KTAS, max fuel, 45-min reserve, ISA, no wind",NaN,NaN,NaN,validation_reference
13,TBM960,Daher,TBM 960,TBM,performance,range,1585.0,NM,reported,A,S05,manufacturer,"290 KTAS, max fuel, 45-min reserve, ISA, no wind",NaN,NaN,NaN,validation_reference
14,TBM960,Daher,TBM 960,TBM,performance,range,1440.0,NM,reported,A,S05,manufacturer,"326 KTAS, max fuel, 45-min reserve, ISA, no wind",NaN,NaN,NaN,validation_reference


**Parameter checks**

In [11]:
conflicts = (
    df_raw
    .groupby(
        ["aircraft_id", "parameter"]
    )["value"]
    .nunique()
    .reset_index(name="number_of_values")
)

conflicts = conflicts[
    conflicts["number_of_values"] > 1
]

conflicts

,aircraft_id,parameter,number_of_values
38,TBM960,range,3


**Clear Dataset**

In [13]:
from tbmx.data.clean import standardize

df = standardize(df_raw)

print("Standardization completed.")
print(df.head())

Standardization completed.
  aircraft_id manufacturer  variant family category     parameter   value  \
0      TBM960        Daher  TBM 960    TBM  weights          MTOW  3454.0   
1      TBM960        Daher  TBM 960    TBM  weights           MRW  3470.0   
2      TBM960        Daher  TBM 960    TBM  weights          MZFW  2836.0   
3      TBM960        Daher  TBM 960    TBM  weights  empty_weight  2180.0   
4      TBM960        Daher  TBM 960    TBM     fuel   usable_fuel  1106.0   

  unit    status confidence source_id    source_type                condition  \
0   kg  reported          A       S02  certification             CG-dependent   
1   kg  reported          A       S02  certification             CG-dependent   
2   kg  reported          A       S05   manufacturer                  6252 lb   
3   kg  reported          A       S05   manufacturer  Prestige cabin; 4806 lb   
4    L  reported          A       S05   manufacturer               292 US gal   

   wind  payload  passe

**Unit validation**

In [14]:
from tbmx.data.schema import validate_parameter_units

unit_problems = validate_parameter_units(df)

print(f"Unit problems found: {len(unit_problems)}")

if unit_problems:
    pd.DataFrame(unit_problems)
else:
    print("All parameter-unit combinations are valid.")

Unit problems found: 0
All parameter-unit combinations are valid.


**Master Dataset Creation**

In [15]:
from tbmx.data.build_master import long_to_master

df_master = long_to_master(df)

print("Master dataset created.")
print(f"Aircraft : {df_master.shape[0]}")
print(f"Features : {df_master.shape[1]}")

Master dataset created.
Aircraft : 5
Features : 35


**Master Dataset View**

In [16]:
df_master

,aircraft_id,manufacturer,variant,family,status,confidence,requirement_flag,MRW_kg,MTOW_kg,MZFW_kg,...,takeoff_power_kW,usable_fuel_L,wing_area_m2,wing_loading_kg_m2,power_to_weight_kW_kg,kg_per_kW,aspect_ratio,empty_weight_fraction,fuel_volume_L_per_MTOW_kg,nominal_cruise_time_h
0,TBM960,Daher,TBM 960,TBM,reported,A,validation_reference,3470.0,3454.000000,2836.0,...,667.401385,1106.000000,18.0,191.888889,0.193226,5.175296,9.144939,0.631152,0.320208,4.803030
1,TBM910,Daher,TBM 910,TBM,reported,B,validation_reference,NaN,3353.861984,NaN,...,633.844891,1101.554829,NaN,NaN,0.188990,5.291298,NaN,0.626048,0.328444,NaN
2,TBM940,Daher,TBM 940,TBM,reported,B,validation_reference,NaN,3354.000000,NaN,...,633.844891,1101.554829,NaN,NaN,0.188982,5.291515,NaN,0.625224,0.328430,NaN
3,PC12NGX,Pilatus,PC-12 NGX,PC12,reported,A,validation_reference,NaN,4740.040267,NaN,...,NaN,1521.735537,NaN,NaN,NaN,NaN,NaN,NaN,0.321039,6.217241
4,M600SLS,Piper,M600/SLS,M600,reported,A,validation_reference,NaN,2721.554220,NaN,...,447.419923,984.207064,NaN,NaN,0.164399,6.082774,NaN,0.608333,0.361634,6.051095


**Features in master dataset**

In [17]:
engineering_features = [
    "aircraft_id",
    "MTOW_kg",
    "empty_weight_kg",
    "range_km",
    "max_cruise_speed_m_s",
    "takeoff_power_kW",
    "wing_loading_kg_m2",
    "power_to_weight_kW_kg",
    "aspect_ratio",
    "empty_weight_fraction"
]

df_master[
    [
        column
        for column in engineering_features
        if column in df_master.columns
    ]
]

,aircraft_id,MTOW_kg,empty_weight_kg,range_km,max_cruise_speed_m_s,takeoff_power_kW,wing_loading_kg_m2,power_to_weight_kW_kg,aspect_ratio,empty_weight_fraction
0,TBM960,3454.000000,2180.000000,2935.420,169.766667,667.401385,191.888889,0.193226,9.144939,0.631152
1,TBM910,3353.861984,2099.679081,NaN,167.708889,633.844891,NaN,0.188990,NaN,0.626048
2,TBM940,3354.000000,2097.000000,NaN,NaN,633.844891,NaN,0.188982,NaN,0.625224
3,PC12NGX,4740.040267,NaN,3339.156,149.188889,NaN,NaN,NaN,NaN,NaN
4,M600SLS,2721.554220,1655.612151,3070.616,140.957778,447.419923,NaN,0.164399,NaN,0.608333


**Master dataset saving...**

In [18]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MASTER_PATH = (
    PROCESSED_DIR
    / "TBM_X_Master_v0_2.csv"
)

df_master.to_csv(
    MASTER_PATH,
    index=False
)

print("Master dataset saved:")
print(MASTER_PATH)

Master dataset saved:
C:\Users\musta\Documents\RAYMER\TBM-X_v0.2_data_engineering\data\processed\TBM_X_Master_v0_2.csv


TBM_X_Benchmark_Dataset_Seed_v0_4.csv
                │
                ▼
           RAW DATA
                │
                ▼
       Schema Validation
                │
                ▼
          Data Cleaning
                │
                ▼
        Unit Validation
                │
                ▼
        Unit Normalization
                │
                ▼
       Long → Master Table
                │
                ▼
      Engineering Features
                │
                ▼
     TBM_X_Master_v0_2.csv